In [17]:
import re
from pathlib import Path
import json
import pandas as pd 

def find_metrics(folder):
    sious = [x for x in Path(folder).rglob('*SIoU*.json')]
    return sious

def read_json(json_path):    
    with open(json_path, 'r') as file:
        # Read the content of the file
        content = file.read()
        # Replace single quotes with double quotes
        content = content.replace("'", '"')
        # Load the JSON data
        data = json.loads(content)
        precision = data.get('precision')
        recall = data.get('recall')
        f1_score = data.get('f1_score')
        
    seed, LR, BS = find_seed_lr_bs(json_path.as_posix())        
    
    return {'precision': precision, 'recall': recall, 'f1_score': f1_score, 'seed': seed, 'LR': LR, 'BS': BS}

def read_json2(json_path):    
    with open(json_path, 'r') as file:
        # Read the content of the file
        content = file.read()
        # Replace single quotes with double quotes
        content = content.replace("'", '"')
        # Load the JSON data
        data = json.loads(content)
        precision = data.get('precision')
        recall = data.get('recall')
        f1_score = data.get('f1_score')
        
    seed, LR, BS = json_path.parent.name.split('_'), json_path.parent.parent.parent.parent.name.split('_')[1], json_path.parent.parent.parent.parent.parent.name.split('_')[1]
    
    return {'precision': precision, 'recall': recall, 'f1_score': f1_score, 'seed': seed, 'LR': LR, 'BS': BS}

def find_seed_lr_bs(path):
    """
    Extracts the seed, learning rate (LR), and batch size (BS) from a given file path.

    The function uses a regular expression to match patterns in the file path that represent seed, LR, and BS.
    It is designed to handle various path structures where these values might appear.

    Parameters:
    path (str): The file path from which to extract the seed, LR, and BS.

    Returns:
    tuple or None: A tuple containing the extracted seed (str), LR (str), and BS (str) if found; 
                   otherwise, None if the pattern does not match.
    """
    # Regular expression to extract seed, LR, and BS from the path
    pattern = r"(?P<seed>\d+)(_BS_(?P<BS>\d+))?_LR_(?P<LR>0\.\d+)/"

    # Search for the pattern in the provided path
    match = re.search(pattern, path)
    if match:
        # Extract seed, LR, and BS (if present)
        seed = match.group('seed')
        BS = match.group('BS') if match.group('BS') else None  # Handle optional BS
        LR = match.group('LR')
        return seed, LR, BS
    else:
        print("No match found")
        return None 
    

files = find_metrics(f'/Data_large/marine/PythonProjects/MMDET/Deploy_out/Sentinel/Export')
print('Len files:', len(files))

res = [read_json2(x) for x in files]
df = pd.DataFrame(res)
grouped_df = df.groupby(['LR', 'BS']).agg({'precision': ['mean', 'std'], 'recall': ['mean', 'std'], 'f1_score': ['mean', 'std']})

print(grouped_df)

Len files: 12
          precision              recall            f1_score          
               mean       std      mean       std      mean       std
LR     BS                                                            
0.0005 2   0.635239  0.052473  0.814516  0.042447  0.713523  0.048329
       3   0.591420  0.036174  0.784946  0.059868  0.674300  0.042426
       4   0.507545  0.161796  0.720430  0.231454  0.591536  0.186430


##### VENUS

In [19]:
files = find_metrics(f'/Data_large/marine/PythonProjects/MMDET/Deploy_out/VENuS/Export')
print('Len files:', len(files))

res = [read_json2(x) for x in files]
df = pd.DataFrame(res)
grouped_df = df.groupby(['LR', 'BS']).agg({'precision': ['mean', 'std'], 'recall': ['mean', 'std'], 'f1_score': ['mean', 'std']})

print(grouped_df.sort_values(by=('f1_score', 'mean'), ascending=False))

Len files: 15
          precision              recall            f1_score          
               mean       std      mean       std      mean       std
LR     BS                                                            
0.0009 3   0.795531  0.025850  0.844379  0.031258  0.819009  0.023945
       4   0.629079  0.264498  0.699882  0.309111  0.662064  0.284839
       2   0.643769  0.228065  0.669586  0.246714  0.656355  0.237098
